In [6]:
import io
import json
import torch
import torch.nn as nn
import pyarrow.dataset as ds

from torch.utils.data import Dataset, DataLoader
from torch.utils.data import IterableDataset
from typing import Literal
from pathlib import Path
from PIL import Image
from torchvision.io import read_image
from tqdm.auto import tqdm
from transformers import BlipProcessor, BlipForQuestionAnswering

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_json(path: Path | str) -> dict:
    with Path(path).open() as f:
        return json.load(f)

In [ ]:
# fp16 instead of fp32, see https://huggingface.co/Salesforce/blip-vqa-base
processor = BlipProcessor.from_pretrained("ybelkada/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("ybelkada/blip-vqa-base", torch_dtype=torch.float16).to(DEVICE)

In [13]:
class BLIPDataset(Dataset):
    def __init__(self, questions_path: Path, images_dir: Path, dataset: Literal['vqa', 'gqa']):
        if dataset == 'vqa':
            self.questions = self._load_vqa_questions(questions_path, images_dir, 'COCO_test2015_')
        elif dataset == 'gqa':
            self.questions = self._load_gqa_questions(questions_path, images_dir)

    def _load_vqa_questions(self, questions_path: Path, images_dir: Path, image_filename_prefix: str):
        return [
            {
                "question_id": int(q["question_id"]),
                "question": q["question"],
                "image_path": images_dir / f"{image_filename_prefix}{q['image_id']:012d}.jpg"
            }
            for q in load_json(questions_path)["questions"]
        ]

    def _load_gqa_questions(self, questions_path: Path, images_dir: Path):
        return [
            {
                "question_id": question_id,
                "question": q["question"],
                "image_path": images_dir / f"{q["imageId"]}.jpg"
            }
            for question_id, q in load_json(questions_path).items()
        ]
    
    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return self.questions[idx]
        

def make_collate_fn(vision_processor, tokenizer):
    def collate_fn(batch):
        question_ids = [b["question_id"] for b in batch]
        questions = [b["question"] for b in batch]
        image_paths = [b["image_path"] for b in batch]

        images = [Image.open(p).convert("RGB") for p in image_paths]
        # images = [read_image(p)[:3] for p in image_paths]

        pixel_values = vision_processor(
            images=images,
            return_tensors="pt"
        )["pixel_values"]
        
        text_inputs = tokenizer(
            questions,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        return {
            "question_ids": question_ids,
            "pixel_values": pixel_values,
            "input_ids": text_inputs["input_ids"],
            "attention_mask": text_inputs["attention_mask"].bool(),
        }

    return collate_fn


@torch.inference_mode()
def compute_predictions(model, processor, dataset, output_path: Path, batch_size: int = 128):
    model.to(DEVICE).eval()
    
    collate_fn = make_collate_fn(processor.image_processor, processor.tokenizer)
    loader = DataLoader(
        dataset,
        batch_size=128,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True,
        prefetch_factor=4
    )

    predictions = []
    
    print('Started inference')

    image_attention_mask = torch.ones((batch_size, 197), device=DEVICE, dtype=torch.bool)

    for batch in tqdm(loader, desc=f"Looping testset"):
        pixel_values = batch["pixel_values"].to(DEVICE, torch.float16, non_blocking=True)
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        image_attention_mask = torch.ones(
            (input_ids.size(0), 197), # images 224x224, 224 / 16 (patch size) = 196 + 1 (CLS token)
            device=DEVICE,
            dtype=torch.bool
        )

        vision_outputs = model.vision_model(pixel_values=pixel_values, return_dict=True)
        
        image_embeds = vision_outputs.last_hidden_state

        question_outputs = model.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True
        )
        
        decoder_input_ids = torch.full(
            (input_ids.size(0), 1),
            model.decoder_start_token_id,
            device=DEVICE,
            dtype=torch.long
        )

        outputs = model.text_decoder.generate(
            input_ids=decoder_input_ids,
            encoder_hidden_states=question_outputs.last_hidden_state,
            max_new_tokens=10,
            eos_token_id=processor.tokenizer.sep_token_id,
            pad_token_id=processor.tokenizer.pad_token_id
        )
        
        answers = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for question_id, answer in zip(batch['question_ids'], answers):
            predictions.append({
                "question_id": question_id,
                "answer": answer
            })

    with open(output_path, 'w') as f:
        json.dump(predictions, f)
    
    print(f"Saved {len(predictions)} predictions to {output_path}")

# VQA eval

In [ ]:
vqa_dataset = BLIPDataset(
    Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_test2015_questions.json'),
    Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Images/test2015'),
    'vqa'
)

compute_predictions(
    model,
    processor,
    vqa_dataset,
    Path('blip_vqa_predictions.json'),
    128
)

In [16]:
gqa_dataset = BLIPDataset(
    Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/testdev_balanced_questions.json'),
    Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA images/images'),
    'gqa'
)

compute_predictions(
    model,
    processor,
    gqa_dataset,
    Path('blip_gqa_predictions.json'),
    128
)

def compute_gqa_accuracy(predictions_path: Path, questions_path: Path, output_path: Path):
    predictions = load_json(predictions_path)
    ground_truth = load_json(questions_path)
    
    binary_correct = 0
    binary_total = 0

    other_correct = 0
    other_total = 0

    total_correct = 0
    total = 0

    for pred in tqdm(predictions):
        qid = pred["question_id"]
        pred_answer = pred["answer"]

        if qid not in ground_truth:
            continue

        gt = ground_truth[qid]

        gt_answer = gt["answer"]
        is_binary = gt_answer in {'yes', 'no'}

        correct = pred_answer == gt_answer

        total += 1

        if correct:
            total_correct += 1

        if is_binary:
            binary_total += 1
            if correct:
                binary_correct += 1

        else:
            other_total += 1
            if correct:
                other_correct += 1

    overall_acc = total_correct / total if total > 0 else 0

    binary_acc = (
        binary_correct / binary_total
        if binary_total > 0 else 0
    )

    other_acc = (
        other_correct / other_total
        if other_total > 0 else 0
    )

    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Binary Accuracy: {binary_acc:.4f}")
    print(f"Other Accuracy: {other_acc:.4f}")

    with open(output_path, "w") as f:
        json.dump({
            "overall": overall_acc,
            "binary": binary_acc,
            "other": other_acc
        }, f)

    print(f"Saved GQA stats to {output_path}")


compute_gqa_accuracy(
    Path('blip_gqa_predictions.json'), 
    Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/testdev_balanced_questions.json'),
    Path('vanilla_gqa.json')
)

  0%|          | 0/12578 [00:00<?, ?it/s]

Overall Accuracy: 0.4711
Binary Accuracy: 0.6566
Other Accuracy: 0.3669
Saved GQA stats to vanilla_gqa.json
